In [9]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# res='1km'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [10]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [11]:
#INITIALIZE DATA FUNCTION
###############################################################
def initiate_array(out_file,vars,t_chunk_size,z_chunk_size,t_size=None,z_size=None):
    # Define array dimensions (adjust based on your data)

    if t_size==None:
        t_size = len(data['time'])  # Number of timesteps
    if z_size==None:
        z_size = len(data['zh'])    # Number of vertical levels
    
    with h5py.File(out_file, 'w') as f: 
        # Check if the dataset 'theta_e' already exists
        for var_name in vars:
            if var_name not in f:
                # Create a dataset with the full size for all time steps (initially empty)
                f.create_dataset(var_name, 
                                 (t_size, z_size),  # Full size for all timesteps
                                 chunks=(t_chunk_size, z_chunk_size))  # Chunks for time axis to allow resizing

In [12]:
# # #TESTING W (T,Z) PLOT
# # w_tz=data['winterp'].mean(dim=("xh",'yh'))

# # w_masked = data['winterp'].where((data['winterp'] >= 0.1) & (data['qc'] + data['qi'] >= 1e-6))
# # w_tz_A = w_masked.mean(dim=("xh", "yh"))  # Compute horizontal mean ignoring NaNs

# # New Subplots for Contour Plots
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# # First contour plot
# cf1 = axes[0].contourf(w_tz.T)
# plt.colorbar(cf1, ax=axes[0],label='m/s')
# axes[0].set_ylabel('zgrid')
# axes[0].set_xlabel('time')
# axes[0].set_title('W(z,t) Horizontal Average for Eulerian Data')

# # Second contour plot
# cf2 = axes[1].contourf(w_tz_A.T)
# plt.colorbar(cf2, ax=axes[1],label='m/s')
# axes[1].set_ylabel('zgrid')
# axes[1].set_xlabel('time')
# axes[1].set_title('W(z,t) Horizontal Average for Eulerian Data (w ≥ 0.1) & (qc+qi ≥ 1e-6)')
# plt.tight_layout()

In [13]:
# #TESTING W (T,Z) PLOT

# z_lev=10
# #GETTING W AND A AT Z LEVEL
# W_z = W[np.where(Z==15)]
# A_z = A_c[np.where(Z==15)]

# #REMOVING ZERO
# W_z = W_z[np.where(A_z!=0)]
# A_z = A_z[np.where(A_z!=0)]

# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# # First subplot
# axes[0].hist(W_z, bins=30, color='blue', alpha=0.7)
# axes[0].set_title(f"W (z = {data['zh'][z_lev].values*1e3:.0f} m)")
# # Second subplot
# axes[1].hist([a * w for a, w in zip(A_z, W_z)], bins=30, color='blue', alpha=0.7)
# axes[1].set_title(f"W * A (z = {data['zh'][z_lev].values*1e3:.0f} m)")

# axes[0].set_ylabel('count');axes[1].set_ylabel('count')
# axes[0].set_xlabel('W (m/s)');axes[1].set_xlabel('W*A (m/s)')
# plt.suptitle('Plotting W array as well as W*A_c for eulerian W for lagrangian parcel at a specific time')
# plt.tight_layout()
# # Lx=512*1000;Ly=200*1000
# # np.sum(A_z*W_z)*np.mean(m_arr)/Lx/Ly/(data['zh'][z_lev].values-data['zh'][z_lev-1].values)/1000


In [14]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=180 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(data['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 0, end_job = 4


In [15]:
#Indexing Array with JobArray
data=data.isel(time=slice(start_job,end_job))
parcel=parcel.isel(time=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [16]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [17]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g', 'A_c', 'W', 'Z', 'Y', 'X']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, W, Z, Y, X = (data_dict[k] for k in var_names)

# #Making Time Matrix
Nt=len(data['time'])
T = np.broadcast_to(np.arange(Nt)[:, None], A_c.shape)  # shape: (Nt, p)

check_memory(globals())

Top 10 objects with highest memory usage
{'W': '810.45 MB', 'Z': '405.22 MB', 'Y': '405.22 MB', 'X': '405.22 MB', 'A_g': '202.61 MB', 'A_c': '202.61 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'FuncAnimation': '0.0 MB'}

2.43133 GB in use overall


In [10]:
# #READING BACK IN
# mins_thresh=5
# dir3=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_{res}_{t_res}_{Np_str}.h5'

# var_names = ['A_g', 'A_c']
# data_dict = make_data_dict(var_names,read_type)
# A_g_Processed, A_c_Processed = (data_dict[k] for k in var_names)
# check_memory()

In [68]:
#############################################################################
#############################################################################

In [18]:
def VMF2d(A, T, Z):
    start_time = time.time()
    """
    Function to compute 2D Mass Flux and update result array based on provided inputs.
    
    Returns a 2D (t,z) array containing the sum of the D array represented by parcels in cloudy updrafts by 1.
    The finally array is then ordered by the appropiate index using the np.add.at function.
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.

    """
    # Compute the difference between neighboring elements along the first axis
    D = A * W
    
    # # Update D for entrainment/detrainment
    # if type=='e':
    #     D[D <= 0] = 0
    # elif type=='d':
    #     D[D >= 0] = 0
    
    # Initialize time and vertical dimension arrays
    Nt = len(data['time']); Nz = len(data['zh'])
    
    # Initialize result array
    result = np.zeros((Nt, Nz))
    
    # Use np.add.at to accumulate values in the result array
    np.add.at(result, (T, Z), D)
    
    end_time = time.time()
    print(f"Execution time: {(end_time - start_time)} seconds")
    return result

In [70]:
# #TESTING TESTING TESTING
# from collections import defaultdict
# def find_repeated_columns(arr):
#     """
#     Given a 2D array of shape (4, N), find indices of columns that repeat (are identical).
#     Returns a list of index arrays, where each array contains indices of repeated columns.
#     """
#     arr_T = arr.T  # shape becomes (N, 4)
    
#     # Dictionary to store unique rows and their corresponding indices
#     row_dict = defaultdict(list)
    
#     for idx, row in enumerate(arr_T):
#         row_tuple = tuple(row)
#         row_dict[row_tuple].append(idx)
    
#     # Return only groups with repeated columns (length >= 2)
#     repeated_indices = [np.array(indices) for indices in row_dict.values() if len(indices) > 1]
    
#     return repeated_indices

# # Example usage
# arr1 = np.array([1, 2,2, 3, 4, 4, 5, 6,6,6])
# arr2 = np.array([1, 3,3, 3, 4, 4, 5, 6,6,6])
# arr3 = np.array([1, 4,4, 3, 4, 4, 5, 6,6,6])
# arr4 = np.array([1, 5,5, 3, 4, 4, 5, 6,6,6])
# arr = np.array([arr1, arr2, arr3, arr4])

# result = find_repeated_columns(arr)
# print(result)


[array([1, 2]), array([4, 5]), array([7, 8, 9])]


In [71]:
# #TESTING TESTING TESTING #*#*
# where=np.where(A==1)
# arr1=T[where]
# arr2=Z[where]
# arr3=Y[where]
# arr4=X[where] 
# arrs=np.array([arr2,arr3,arr4]) 
# arrs

# out=find_repeated_columns(arrs)
# print(out)

# # one=where[0][1],where[1][1]
# # two=where[0][2],where[1][2]
# # one,two
# # T[one],Z[one],Y[one],X[one]
# # T[two],Z[two],Y[two],X[two]

In [72]:
# #TESTING TESTING TESTING
# def apply_function(A):
#     # A1=A.copy() #TESTING

#     where=np.where(A==1)
#     arr1=T[where]
#     arr2=Z[where]
#     arr3=Y[where]
#     arr4=X[where] 
#     arr=np.array([arr1,arr2,arr3,arr4]) 

#     out = find_repeated_columns(arr)
#     # print(out)

#     for ind in np.arange(len(out)):
#         extras=out[ind][1:]
#         A[(where[0][extras],where[1][extras])]=0

#     # A2=A.copy() #TESTING
#     # print(np.all(A1!=A2)) #TESTING

#     #TESTING 
#     # for k in range(len(out)):
#     #     ind1=out[k][0]
#     #     ind2=out[k][1]
#     #     print((arr1[ind1],arr2[ind1],arr3[ind1],arr4[ind1])==(arr1[ind2],arr2[ind2],arr3[ind2],arr4[ind2]))
#     return A

# # apply_function(A) #TESTING

In [19]:
#TURN PROCESSING ON OR OFF
PROCESSING=False
# PROCESSING=True

# Set A based on PROCESSING state
print('Calculating 2D VMF for General Updrafts')
A = A_g if (PROCESSING==False) else A_g_Processed
# A=apply_function(A) #TESTING TESTING TESTING
profile_array_VMF_g = VMF2d(A, T, Z)


# Set A for the second block
print('Calculating 2D VMF for Cloudy Updrafts')
A = A_c if (PROCESSING==False) else A_c_Processed
# A=apply_function(A) #TESTING TESTING TESTING
profile_array_VMF_c = VMF2d(A, T, Z)

Calculating 2D VMF for General Updrafts
Execution time: 16.734865188598633 seconds
Calculating 2D VMF for Cloudy Updrafts
Execution time: 16.552248001098633 seconds


In [20]:
dir2=dir+'Project_Algorithms/Entrainment/job_out_4/'

#SAVING
if PROCESSING==False:
    out_file=dir2+f'2D_VMF_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
elif PROCESSING==True:
    out_file=dir2+f'2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'


vars=["profile_array_VMF_g","profile_array_VMF_c"]
initiate_array(out_file,vars,t_chunk_size=1,z_chunk_size=17)

with h5py.File(out_file, 'a') as f: 
    f['profile_array_VMF_g'][:]=profile_array_VMF_g
    f['profile_array_VMF_c'][:]=profile_array_VMF_c
print('done')

done


In [21]:
check_memory(globals())

Top 10 objects with highest memory usage
{'W': '810.45 MB', 'Z': '405.22 MB', 'Y': '405.22 MB', 'X': '405.22 MB', 'A_g': '202.61 MB', 'A_c': '202.61 MB', 'A': '202.61 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB'}

2.63394 GB in use overall


In [22]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE DURING JOB ARRAY RUN
# recombine=True

In [23]:
if recombine==True:
    PROCESSING=False
    # PROCESSING=True
    
    dir2=dir+'Project_Algorithms/Entrainment/job_out_4/'
    dir3=dir+'Project_Algorithms/Entrainment/'
    
    if PROCESSING==False:
        out_file=dir3+f'2D_VMF_profiles_{res}_{t_res}_{Np_str}.h5'
    elif PROCESSING==True:
        out_file=dir3+f'2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
    
    vars=["profile_array_VMF_g","profile_array_VMF_c"]
    initiate_array(out_file,vars,t_chunk_size=50,z_chunk_size=17)
    
    with h5py.File(out_file, 'r+') as f_out:
        num_jobs=60
        for job_id in np.arange(1,num_jobs+1):
            if np.mod(job_id,5)==0: print(f"job_id = {job_id}")
            [a,b] = get_job_range(job_id,num_jobs)
    
            if PROCESSING==False:
                in_file=dir2+f'2D_VMF_profiles_{res}_{t_res}_{Np_str}_{job_id}.h5'
            elif PROCESSING==True:
                in_file=dir2+f'2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}_{job_id}.h5'
            with h5py.File(in_file, 'r') as f_in: 
                for var in vars:
                    f_out[var][a:b]=f_in[var][:]

In [24]:
########################################################
#PLOTTING
plotting=False #KEEP FALSE DURING JOB ARRAY RUN
# plotting=True

In [25]:
if plotting==True:
    #constants
    Cp=1004 #Jkg-1K-1
    Cv=717 #Jkg-1K-1
    Rd=Cp-Cv #Jkg-1K-1
    eps=0.608
    
    Lx=(data['xf'][-1].item()-data['xf'][0].item())*1000 #x length (m)
    Ly=(data['yf'][-1].item()-data['yf'][0].item())*1000 #y length (m)
    Np=len(parcel['xh']) #number of lagrangian parcles
    dt=(data['time'][1]-data['time'][0]).item()/1e9 #sec
    dx=(data['xf'][1].item()-data['xf'][0].item())*1e3 #meters
    dy=(data['yf'][1].item()-data['yf'][0].item())*1e3 #meters
    xs=data['xf'].values*1000
    ys=data['yf'].values*1000
    zs=data['zf'].values*1000
    
    def zf(z):
        k=z #z is the # level of z
        out=data['zf'].values[k]*1000
        
        return out
    # def rho(x,y,z,t):
    #     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
    #     p0=101325 #Pa
    #     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
    #     T=theta*(p/p0)**(Rd/Cp)
    #     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
    #     # Tv=T*(1+eps*qv)
    #     Tv=T*(eps+qv)/(eps*(1+qv))
    #     rho = p/(Rd*Tv)
    #     out=rho
    #     return out
    
    def rho(x,y,z,rho_data_t):
        out=rho_data_t[z,y,x]
        return out
    def m(t):
        rho_data_t=data['rho'].isel(time=t).data
        
        m=0
        #triple sum
        for k in range(len(data['zh'])):
            dz=(zf(k+1)-zf(k))
            for j in range(len(data['yh'])):
                for i in range(len(data['xh'])):
                    rho_out=rho(i,j,k,rho_data_t)
                    m+=rho_out*dz
                    
        #triple sum
        out=m*dx*dy/Np
        return out

In [26]:
if plotting==True:
    #Calculate Mass Constant
    # calculate='single_time'
    calculate=False
    
    if calculate==True:
        Nt=len(data['time'])
        m_arr=np.zeros((Nt))
        for t in np.arange(Nt):
            if np.mod(t,25)==0: print(t)
            # m_arr[t]=m(t) #UNCOMMENT FOR FULL CALCULATION
        # np.save('Mass_Array.npy', m_arr)
        np.save('Mass_Array_1min.npy', m_arr)
    elif calculate=='single_time':
        Nt=len(data['time'])
        m_arr=np.zeros((Nt))
    
        t=len(data['time'])//2
        m_300=m(t)
        for t in np.arange(Nt):
            m_arr[t]=m_300 #UNCOMMENT FOR FULL CALCULATION
        # np.save('Mass_Array.npy', m_arr)
        np.save('Mass_Array_1min.npy', m_arr)
    else:
        dir3=dir+f'Project_Algorithms/Entrainment/'
        # m_arr = np.load('Mass_Array.npy')
        m_arr = np.load('Mass_Array_1min.npy')
    
    # # TESTING
    # lst=[]
    # for t in np.arange(133):
    #     lst.append(m_arr[t])
    
    # plt.plot(lst)
    # (np.max(lst)-np.min(lst))*100/np.mean(lst)

In [27]:
if plotting==True:
    PROCESSING=False
    PROCESSING=True
    
    if PROCESSING==False:
        dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_{res}_{t_res}_{Np_str}.h5'
    if PROCESSING==True:
        dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(dir3, "r") as h5f:
        profile_array_VMF_g = h5f["profile_array_VMF_g"][:]
        profile_array_VMF_c = h5f["profile_array_VMF_c"][:]

In [28]:
if plotting==True:
    def apply_constant(profile_array,apply):
        if apply==True:
            Nt=profile_array.shape[0]
            Nz=profile_array.shape[1]
        
            profile_array/=(Lx*Ly)
            for t in np.arange(Nt):
                profile_array[t]*=m_arr[t]
            for z in np.arange(Nz):
                dz=zf(z+1)-zf(z)
                profile_array[:,z]/=dz
        return profile_array
    
    profile_array_VMF_g=apply_constant(profile_array_VMF_g,apply=True)
    profile_array_VMF_c=apply_constant(profile_array_VMF_c,apply=True)

In [29]:
if plotting==True:
    def VMF_Profile(ax,t,data):
            
        
        VMF_z = profile_array_VMF_c[t]
        #########################
    
        
        #First plot
        ax.plot(VMF_z,data['zh'])
        ax.axvline(0,color='black')
        ax.set_xlabel(r"$kg m^{-2} s^{-1}$")
        ax.set_title(f'Horizontal Average of 2D VMF For Entire Domain  (AT TIME {t})',fontsize=8)
        
        apply_scientific_notation([ax])
    
        ax.set_ylim(bottom=0)
        return axes
    
    
    fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(15, 10))
    axes = axes.flatten()
    times = np.arange(0, 130, 10)
    
    for idx, t in enumerate(times):
        VMF_Profile(axes[idx], t, data)
    
    plt.tight_layout()
    plt.ylabel('z (km)')

In [30]:
if plotting==True:
    type='general'
    type='cloudy'
    
    if type=='general':
        profile_array=profile_array_VMF_g
    if type=='cloudy':
        profile_array=profile_array_VMF_c
    
    
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    import numpy as np
    
    fig = plt.figure(figsize=(10, 8))
    gs = GridSpec(2, 2, figure=fig)
    
    ######
    cmap1 = plt.cm.viridis
    cmap2 = plt.cm.seismic 
    n_levels=29
    ######
    
    # ######
    # vmax_shared = np.max([np.max(profile_array_e), np.max(profile_array_d)])
    # norm_shared = mcolors.Normalize(vmin=0, vmax=vmax_shared)
    # ######
    
    # First subplot: VMF
    ########################################
    ax1 = fig.add_subplot(gs[0, 0])
    # contour1 = ax1.contourf(profile_array_e.T, cmap=cmap1)
    contour1 = ax1.contourf(profile_array.T, cmap=cmap1)
    cbar1=fig.colorbar(contour1, ax=ax1)
    apply_scientific_notation_colorbar([cbar1])
    Nz = len(data['zh'])
    ax1.set_yticks(np.arange(Nz))
    new_ytick_labels = np.round(data['zf'].values[:Nz], 2)
    ax1.set_yticklabels(new_ytick_labels, fontsize=8, rotation=0)
    ax1.set_ylabel('z (km)');ax1.set_xlabel('t (timesteps)')
    ax1.set_title('Entrainment using Lagrangian Binary Array',fontsize=8)
    
    # #FIXING TICKS
    # ax3.set_yticks(np.arange(Nz))
    # new_ytick_labels = np.round(data['zf'].values[:Nz], 2)
    # ax3.set_yticklabels(new_ytick_labels, fontsize=8, rotation=0)
    # ax3.set_ylabel('z (km)');ax3.set_xlabel('t (timesteps)')
    # ax3.set_title('Entrainment - Detrainment')
    
    # #FIXING SCIENTIFIC NOTATION
    # from matplotlib.ticker import ScalarFormatter
    # formatter = ScalarFormatter(useMathText=True)
    # formatter.set_powerlimits((-2, 2))  # Adjust the range for scientific notation
    # for cbar in (cbar1,cbar2, cbar3):  # These must be Colorbar instances
    #     cbar.formatter = formatter
    #     cbar.update_ticks()
    
    # Display the plot
    plt.tight_layout()

In [31]:
if plotting==True:
    plt.plot(np.mean(profile_array,axis=(0)),data['zh'],label='VMF')
    
    plt.legend(); plt.title('2D Vertical Mass Flux Using Lagrangian Binary Array')
    
    from matplotlib.ticker import ScalarFormatter
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-1, 1))
    plt.gca().xaxis.set_major_formatter(formatter)
    
    plt.ylabel('z (km)');plt.xlabel('Vertical Mass Flux, M(t,z) ' + r'$(kg m^{-2} s^{-1})$')
    
    # #COMPARING MAXIMUM WITH ROMPS 2012
    # (1.4e-2)/(4e-3)

In [32]:
# #TESTING DISTRIBUTION

# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # Create a 1-row, 3-column subplot

# array_z=np.mean(profile_array,axis=(0)); where=np.where(array_z==np.max(array_z))[0] #12
# profile_array_z=profile_array.copy()[:,where]

# # Plot the first histogram for all values
# axes[0].hist(profile_array_z)
# axes[0].set_ylabel('Count')
# axes[0].set_xlabel('Vertical Mass Flux, M(z) ' + r'$(kg m^{-2} s^{-1})$')
# axes[0].set_title('Histogram of VMF(t,z)')

# # Threshold-based filtering
# threshold = 0.01  
# filtered_data_below = profile_array_z[profile_array_z < threshold]
# filtered_data_above = profile_array_z[profile_array_z > threshold]

# # Plot the second histogram for values < 0.01
# axes[1].hist(filtered_data_below)
# axes[1].set_ylabel('Count')
# axes[1].set_xlabel('Vertical Mass Flux, M(z) ' + r'$(kg m^{-2} s^{-1})$')
# axes[1].set_title('Histogram of VMF(t,z) for values < 0.01')

# # Plot the third histogram for values > 0.01
# axes[2].hist(filtered_data_above)
# axes[2].set_ylabel('Count')
# axes[2].set_xlabel('Vertical Mass Flux, M(z) ' + r'$(kg m^{-2} s^{-1})$')
# axes[2].set_title('Histogram of VMF(t,z) for values > 0.01')

# plt.suptitle(f"Histograms of VMF at z = {data['zh'].values[where]} km (the location of the peak in VMF(t,z)")
# plt.tight_layout()  # Adjust layout to prevent overlap

# f"{(0.0005*50)*100/(0.03*14):.2f}% contribution for the values below 0.001"